## Structured Output

In [2]:
# Models can be requested to provide thier reponse in a format matching a given schema 


### Pydantic 

In [17]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

In [28]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY")
)
model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-google-genai': '4.3.3'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x000001DA473F7190>, default_metadata=(), model_kwargs={})

In [29]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie out of 10")


In [30]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-google-genai': '4.3.3'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x000001DA473F7190>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 'response_jso

In [31]:
response = model_with_structure.invoke("Tell me about the movie Inception")
response 

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure 

In [32]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw =True)

In [33]:
response = model_with_structure.invoke("Tell me about the movie Inception")
response 

{'raw': AIMessage(content='{"title": "Inception", "year": 2010, "director": "Christopher Nolan", "rating": 8.8}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a05bb3-3409-7541-95ce-fa849f3c2d74-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 91, 'total_tokens': 99, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 60}}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8),
 'parsing_error': None}

### Nested Structure 

In [34]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    director:str
    cast:list[Actor]
    genres : list[str]
    budget:float | None = Field(default=None, description="The budget of the movie in USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Tell me about the movie Inception")
response

MovieDetails(title='Inception', year=2010, director='Christopher Nolan', cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito')], genres=['Science Fiction', 'Action', 'Thriller'], budget=160000000.0)

### TypeDict 

In [35]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "Title of the movie"]
    year : Annotated[int, ..., "Release year of the movie"]
    director : Annotated[str, ..., "Director of the movie"]
    rating : Annotated[float, ..., "Rating of the movie out of 10"]

model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("Please provide the details of the movie Avengers")
response

{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'rating': 8.0}

### DataClasses

In [36]:
import os
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

In [41]:
from dataclasses import dataclass, field
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact Information of a person"""
    name:str # The name of the person
    email : str #Email address of the person 
    phone : str # Phone number of the person

agent = create_agent(
    model = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash"
    ),
    response_format = ContactInfo
)

result = agent.invoke({
    "messages" : [{"role":"user", "content":"Please provide the contact information of John Doe"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john.doe@example.com', phone='123-456-7890')